# 实验一 · 轨迹数据预处理完成版

对应教师 `作业1轨迹数据预处理.ipynb`。原 starter 保留；本完成版复用经过核验的同一套实现，依次展示 S 分段/过滤、D 方向删除、P 简化、三组参数、六种顺序、评价与真实 AI 批判，最后实际全量复算。

**默认 FULL_RECOMPUTE**：历史参数/顺序从 raw 重算 9,720 个记录—候选配对，再从全部 11,386 条原始记录重算冻结 R0/最终策略，并从本次新结果重新生成图与画图数据。`LIVE` 不在本 Notebook 自动开启。历史 G2 结果、最终确认样本、全量生产三者保持独立身份。可用 `SC_LAB1_RECOMPUTE_WORK` 指定工程外的新输出目录；不设置则创建临时目录，不覆盖已有非空结果。

In [1]:
from pathlib import Path
import contextlib, csv, gzip, hashlib, html, io, json, os, platform, re, sys, tempfile
from IPython.display import display, HTML, Image, Markdown

# 从 Notebook 所在位置或启动目录向上找项目；不依赖个人绝对路径和 ROOT/.venv。
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p/'task1/goal3').is_dir()), None)
if ROOT is None:
    raise RuntimeError('请把 Notebook 放在完整提交包中，或从项目目录启动。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = 'FULL_RECOMPUTE'
ENABLE_LIVE = False
assert MODE == 'FULL_RECOMPUTE' and ENABLE_LIVE is False
explicit_work = os.environ.get('SC_LAB1_RECOMPUTE_WORK')
if explicit_work:
    WORK = Path(explicit_work).expanduser().resolve()
    if WORK == ROOT or ROOT in WORK.parents:
        raise RuntimeError('显式复算目录必须位于当前工程/解压包之外。')
    if WORK.exists() and (not WORK.is_dir() or any(WORK.iterdir())):
        raise RuntimeError('显式复算目录已存在且非空，拒绝覆盖。')
    WORK.mkdir(parents=True, exist_ok=True)
else:
    WORK = Path(tempfile.mkdtemp(prefix='sc-lab1-full-')).resolve()

from task1.workflow.io import digest, object_hash, read_json, write_json
from task1.workflow.io import ROOT as MODULE_ROOT
from task1.workflow.g2_provider import ExperimentProvider
from task1.workflow.provider import CodexProvider
from task1.goal3.reproduce import recompute_historical, recompute_production
assert MODULE_ROOT.resolve() == ROOT, '模块必须来自当前工程/解压包，不能借用原工作区。'

# 本内核实际拦截两个已登记 Provider 入口；出现调用立即失败，不回退到 Mock。
provider_observation = {'attempted_new_calls': 0}
def forbid_provider(*args, **kwargs):
    provider_observation['attempted_new_calls'] += 1
    raise RuntimeError('FULL_RECOMPUTE 禁止新增模型调用')
_original_experiment_call = ExperimentProvider.call
_original_governance_call = CodexProvider.call
ExperimentProvider.call = forbid_provider
CodexProvider.call = forbid_provider

G2 = ROOT/'task1/evidence/goal2'
G3 = ROOT/'task1/evidence/goal3'
stage = read_json(G2/'result_summary.json')
memory_path = ROOT/stage['memory']['snapshot_path']
memory_hash_before = digest(memory_path)
assert memory_hash_before == stage['memory']['snapshot_sha256']
raw_path = ROOT/'task1/作业/作业/traj_dict.json'
assert digest(raw_path) == stage['data']['raw_sha256']

# 只有当前生产已冻结并有真实 run 时才执行本完成版，不临时用样本替代 full。
if not (G3/'current_runs.json').is_file() or not (G3/'production_freeze.json').is_file():
    raise RuntimeError('G3 正式生产尚未登记；保持源文件待执行，不把 fast 样本冒充 full。')

def show(rows, columns=None, limit=20):
    rows = list(rows)
    if not rows:
        print('无对应行；不填造数值。')
        return
    columns = columns or list(rows[0])
    def cell(value):
        if value is None: return '不可用（见相应原因）'
        if isinstance(value, (dict,list)): value=json.dumps(value,ensure_ascii=False)
        return html.escape(str(value))
    head='<tr>'+''.join('<th>'+cell(c)+'</th>' for c in columns)+'</tr>'
    body=''.join('<tr>'+''.join('<td>'+cell(r.get(c))+'</td>' for c in columns)+'</tr>' for r in rows[:limit])
    display(HTML('<table style="border-collapse:collapse" border="1">'+head+body+'</table>'))
    if len(rows)>limit: print(f'显示前 {limit} / {len(rows)} 行；完整记录保留于来源文件。')

def figure(name, phase='goal2', width=950):
    path=ROOT/'task1/figures'/phase/(name+'.png')
    if not path.is_file(): raise FileNotFoundError('正式图不存在：'+str(path.relative_to(ROOT)))
    display(Markdown('**已归档正式图：** '+phase+' / '+name+'（历史解释材料）'))
    display(Image(filename=str(path),width=width))

def recomputed_figure(topic, name, width=950):
    directory=WORK/topic/'figures'
    manifest=read_json(directory/'figure_manifest.json')
    assert manifest['old_summary_or_figure_read_for_plotting'] is False
    entry=next(f for f in manifest['figures'] if f['name']==name)
    path=directory/entry['files']['png']['path']
    assert digest(path)==entry['files']['png']['sha256']
    assert digest(directory/'plot_data.json')==manifest['plot_data_sha256']
    display(Markdown('**本次 FULL 从 raw 重新计算并绘制：** '+name))
    display(Image(filename=str(path),width=width))

def run_logged(label, fn):
    capture=io.StringIO()
    with contextlib.redirect_stdout(capture):
        result=fn()
    (WORK/(label+'_progress.txt')).write_text(capture.getvalue(),encoding='utf8')
    assert result['status']=='VERIFIED' and result['new_model_calls']==0
    print({k:v for k,v in result.items() if k not in {'checks','selections'}})
    return result

def current_manifest(partition):
    index=read_json(G3/'current_runs.json')
    found=[]
    for run_id in index.values():
        if not isinstance(run_id,str): continue
        path=G3/'runs'/run_id/'manifest.json'
        if path.is_file():
            value=read_json(path)
            if value.get('partition')==partition: found.append((path,value))
    if len(found)!=1: raise RuntimeError('当前 run 索引没有唯一分区：'+partition)
    return found[0]

_, deployment_manifest=current_manifest('FULL_PRODUCTION')
if (deployment_manifest.get('status')!='MACHINE_VERIFIED_PENDING_C'
        or deployment_manifest['input_ids']!=deployment_manifest['completed_record_ids']):
    raise RuntimeError('当前生产尚未完整完成；不执行待验收Notebook。')

print('FULL_RECOMPUTE；Python',platform.python_version(),'; 新模型调用由实际 Provider 哨兵检查。')
print('数据：',stage['data']['raw_records'],'条 /',stage['data']['raw_points'],'点；source_crs=UNVERIFIED。')

FULL_RECOMPUTE；Python 3.12.3 ; 新模型调用由实际 Provider 哨兵检查。
数据： 11386 条 / 1173410 点；source_crs=UNVERIFIED。


## 1. 数据与条件化坐标

输入原始坐标/时间不覆盖。工作坐标固定椭球 `a=6378137 m, 1/f=298.257223563`、`h=0`、ENU 中心 `(121.343555°,31.3561015°)`。数值核对不能证明源 datum；零时间差速度、零位移方向分别记为不可计算。以下只用已暴露的 pilot 记录 246 演示单条步骤，不从最终确认集挑案例。

In [2]:
from task1.workflow.g2_data import raw_data, adapt
from task1.workflow.g2_pipeline import run_record, REFERENCE_PARAMETERS, DIRECTION_METHOD
from task1.workflow.g2_metrics import review_record
from task1.workflow.geometry import (split_trajectory, filter_segments, denoise_trajectory,
                                    douglas_peucker_indices, select_indices, recompute_features)
raw=raw_data()
pilot=adapt('246',raw['246'])
print('演示身份：已暴露 PILOT 246；原始点数',len(pilot['indices']))
print('教学参考参数',REFERENCE_PARAMETERS)
features=recompute_features(pilot,time_reliable=True)
show(features['edges'][:8],['from_index','to_index','dt','distance','speed','speed_reason','direction_degrees','direction_reason'])

演示身份：已暴露 PILOT 246；原始点数 108
教学参考参数 {'dt': 30, 'distance': 400, 'min_points': 5, 'min_length': 65, 'direction': 35, 'dp': 5}


from_index,to_index,dt,distance,speed,speed_reason,direction_degrees,direction_reason
0,1,7,1.0997077325179991,0.15710110464542845,不可用（见相应原因）,72.36717872281527,不可用（见相应原因）
1,2,20,1.4965836182500356,0.07482918091250178,不可用（见相应原因）,72.73498112334407,不可用（见相应原因）
2,3,10,0.34599885416285686,0.03459988541628568,不可用（见相应原因）,343.9864201526183,不可用（见相应原因）
3,4,10,0.29239063512813873,0.029239063512813874,不可用（见相应原因）,319.2949351606239,不可用（见相应原因）
4,5,10,0.2858723700163614,0.02858723700163614,不可用（见相应原因）,269.97263003978844,不可用（见相应原因）
5,6,10,0.3969609147782863,0.039696091477828634,不可用（见相应原因）,286.1913057271124,不可用（见相应原因）
6,7,20,0.8646574936546763,0.04323287468273381,不可用（见相应原因）,26.12931223113914,不可用（见相应原因）
7,8,20,76.99950084071651,3.849975042035825,不可用（见相应原因）,293.2997358837098,不可用（见相应原因）


## 2. 完成三个基础 TODO

### S：先断边，再按完整段过滤

`dt > 阈值`、负时间差或距离 `> 阈值` 才断开；等号不切，右点属于新段。短段依据段内累计长度与点数判定，断开的边不计入长度。下面是教师 TODO 对应的可读调用入口，核心数学只有模块内一份。

In [3]:
def segmentation(record, dt=30, distance=400, min_points=5, min_length=65):
    split=split_trajectory(record,dt,distance,time_reliable=True)
    filtered=filter_segments(split['segments'],min_points,min_length,time_reliable=True)
    return {'boundaries':split['boundaries'],'kept':filtered['kept'],'dropped':filtered['dropped']}

segmented=segmentation(pilot)
print('原始断点:',segmented['boundaries'])
show([{'kept':keep,'original_indices':part['indices'],
       'length_work_m':part['features']['length'],
       'filter_reasons':part.get('filter_reasons',[])}
      for keep,key in [(True,'kept'),(False,'dropped')] for part in segmented[key]])

原始断点: []


kept,original_indices,length_work_m,filter_reasons
True,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107]",11326.480988154068,[]


### D：完整标记后同时删除

当前至后继为出边方向；与前后出边方向的圆周差都严格大于阈值才标记。一次标记、同时删除，不迭代；首尾及必要方向不可计算窗口保留并标记，下游重算特征。没有未知物理限速或“低位移就可靠”的假设。

In [4]:
def denoising(segment,direction=35):
    return denoise_trajectory(segment,direction,method=DIRECTION_METHOD,time_reliable=True)

denoised=[denoising(part) for part in segmented['kept']]
show([{'segment':i,'input_points':len(part['indices']),
       'deleted_indices':result['deleted_indices'],
       'remaining_points':len(result['record']['indices'])}
      for i,(part,result) in enumerate(zip(segmented['kept'],denoised))])

segment,input_points,deleted_indices,remaining_points
0,108,"[6, 20, 48]",105


### P：有限线段的 Douglas–Peucker

把点向有限端点线段投影，投影位置夹在 `[0,1]`；端点重合用点到端点距离。最大偏差严格超过容差时递归。`dp=0` 明确作为全部保留参考。输出沿用原始身份，不插值、不改坐标和时间。

In [5]:
def simplification(segment,dp=5):
    kept=douglas_peucker_indices(segment['xy'],dp,segment['indices'])
    return select_indices(segment,kept,time_reliable=True)

simplified=[simplification(result['record']) for result in denoised]
show([{'segment':i,'P_input':len(clean['record']['indices']),
       'P_output':len(final['indices']),'retained_original_indices':final['indices']}
      for i,(clean,final) in enumerate(zip(denoised,simplified))])

# 已知数学答案用于理解距离定义，非真实数据噪声标签。
from task1.workflow.geometry import point_segment_distance
print('越过端点的点到有限线段距离：',point_segment_distance((2,1),(0,0),(1,0)))

segment,P_input,P_output,retained_original_indices
0,105,36,"[0, 9, 12, 26, 27, 33, 35, 37, 39, 41, 43, 44, 46, 47, 49, 50, 51, 52, 53, 55, 59, 60, 61, 62, 69, 70, 71, 80, 81, 82, 85, 89, 94, 104, 106, 107]"


越过端点的点到有限线段距离： 1.4142135623730951


## 3. 主函数、动作账本与真实前后案例

`run_record` 统一记录每阶段输入/操作/输出，原始点的终态为 filtered、denoised、simplified 或 retained。审核从独立可信 pilot 输入重构，不拿候选自己的 clean 自证。图分别画真实片段，原始断点不重连。

In [6]:
contract_hash=digest(ROOT/'task1/config/goal2/contract.json')
provenance={'source_kind':'CURRENT_NOTEBOOK_EXPOSED_PILOT','record_id':'246','not_new_independent_sample':True}
example=run_record(pilot,dict(REFERENCE_PARAMETERS),'S-D-P',contract_hash=contract_hash,provenance=provenance)
check=review_record(example,pilot,expected_parameters=dict(REFERENCE_PARAMETERS),expected_order='S-D-P',
                    contract_hash=contract_hash,trusted_provenance=provenance)
assert check['status']=='VERIFIED'
show([{k:example['metrics'][k] for k in ['n_input','n_filtered','n_direction_removed','n_dp_removed','n_final',
      'common_covered_points','common_max_error','dp_max_error','raw_break_crossings']}])
show(example['point_actions'][:12],['original_index','action','reasons'])

import matplotlib
# 文件输出与显式显示不依赖 Notebook 前端的 inline 后端版本。
matplotlib.use('Agg')
import matplotlib.pyplot as plt
palette_text=(ROOT/'templates/latex/common/p2_cloud_sorbet_colors.tex').read_text()
palette={name:'#'+value for name,value in re.findall(r'\\definecolor\{([^}]+)\}\{HTML\}\{([0-9A-Fa-f]{6})\}',palette_text)}
raw_windows=split_trajectory(pilot,30,400,time_reliable=True)['segments']
fig,axes=plt.subplots(1,3,figsize=(13,4),constrained_layout=True)
for ax,title,parts,color in zip(axes,['Raw fixed windows','After S-D','After S-D-P'],
        [raw_windows,example['stages'][1]['output'],example['final_segments']],[palette['Muted'],palette['C2'],palette['C1']]):
    for part in parts:
        if part['xy']:
            x,y=zip(*part['xy']);ax.plot(x,y,'.-',color=color,linewidth=1,markersize=3)
    ax.set_title(title);ax.set_xlabel('East / working m');ax.set_ylabel('North / working m')
    ax.set_aspect('equal',adjustable='datalim');ax.grid(alpha=.2)
fig.suptitle('Known-exposed pilot 246; conditional coordinates; no basemap')
fig.savefig(WORK/'pilot_steps.svg')
fig.savefig(WORK/'pilot_steps.png',dpi=200)
plt.close(fig)
display(Image(filename=str(WORK/'pilot_steps.png'),width=1000))

n_input,n_filtered,n_direction_removed,n_dp_removed,n_final,common_covered_points,common_max_error,dp_max_error,raw_break_crossings
108,0,3,69,36,108,111.93999607553866,4.768356861321128,0


original_index,action,reasons
0,retained,"[""FINAL_OUTPUT""]"
1,simplified,"[""DP_WITHIN_TOLERANCE""]"
2,simplified,"[""DP_WITHIN_TOLERANCE""]"
3,simplified,"[""DP_WITHIN_TOLERANCE""]"
4,simplified,"[""DP_WITHIN_TOLERANCE""]"
5,simplified,"[""DP_WITHIN_TOLERANCE""]"
6,denoised,"[""DIRECTION_RULE""]"
7,simplified,"[""DP_WITHIN_TOLERANCE""]"
8,simplified,"[""DP_WITHIN_TOLERANCE""]"
9,retained,"[""FINAL_OUTPUT""]"


ImportError: cannot import name 'backend2gui' from 'IPython.core.pylabtools' (/home/addaswsw/lab/Smart_Cities_and_Location_Services/.venv/lib/python3.12/site-packages/IPython/core/pylabtools.py)

C-only engineering observation below; the six preceding code cells are unchanged actual coursework cells. This prefix run is not FULL_RECOMPUTE completion.

In [ ]:
assert provider_observation['attempted_new_calls'] == 0
assert check['status']=='VERIFIED'
assert matplotlib.get_backend().lower()=='agg'
assert digest(memory_path)==memory_hash_before
write_json(WORK/'independent_smoke_probe.json', {'classification':'REAL_PILOT_PREFIX_SMOKE_NOT_FULL', 'record_id':pilot['record_id'], 'points':len(pilot['indices']), 'backend':matplotlib.get_backend(), 'new_model_calls':provider_observation['attempted_new_calls'], 'trace_review':check['status'], 'memory_unchanged':digest(memory_path)==memory_hash_before, 'actual_metrics':example['metrics']})